# NB_GLD_SRV_POI_INFO

RECOMPUTE `lh_vv_gold.dbo.gld_srv_poi_info`. Grain `poi_id`.
Partner giữ mọi dòng. Google chỉ khi `poi_id` có ở cả `slv_enr_poi_destination` và `slv_enr_poi_info_multi_lang`.
`operating_hours`: Google theo mapping §1.3, partner theo §1.5.
`rating_overall`, `rating_count`, `review_count`: chỉ Google, left join `slv_3p_poi_rating` / `slv_3p_poi_review` theo `poi_id` (mapping §1.2).


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
# BEGIN_HOURS_PURE
import json

DAY_LABELS = {1: "Thứ Hai", 2: "Thứ Ba", 3: "Thứ Tư", 4: "Thứ Năm", 5: "Thứ Sáu", 6: "Thứ Bảy", 7: "Chủ Nhật"}
PARTNER_DAYS = ("monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday")
CLOSED_VALUE = "Đóng cửa"
ALL_DAY = {"open_hour": 0, "open_minute": 0, "day_of_week": 0}
TIME_KEYS = ("open_hour", "open_minute", "close_hour", "close_minute")


def _int(value):
    return isinstance(value, int) and not isinstance(value, bool)


def _open(day, open_time, close_time):
    return {"day_of_week": day, "label": DAY_LABELS[day], "value": f"{open_time} - {close_time}",
            "open_time": open_time, "close_time": close_time, "is_open_all_day": False, "is_closed": False}


def _closed(day):
    return {"day_of_week": day, "label": DAY_LABELS[day], "value": CLOSED_VALUE,
            "is_open_all_day": False, "is_closed": True}


def _all_day(day):
    return {"day_of_week": day, "label": DAY_LABELS[day], "is_open_all_day": True, "is_closed": False}


def _dump(days):
    return json.dumps(days, ensure_ascii=False, separators=(",", ":"))


def google_hours(text):
    if text is None:
        return None, None                                            # rule 8, D-SRV-25
    try:
        doc = json.loads(text)
    except (TypeError, ValueError, RecursionError):
        return None, "GOOGLE_INVALID_JSON"
    if not isinstance(doc, list):
        return None, "GOOGLE_NOT_ARRAY"
    if len(doc) == 1 and isinstance(doc[0], dict) and doc[0] == ALL_DAY and all(_int(v) for v in doc[0].values()):
        return _dump([_all_day(day) for day in range(1, 8)]), None   # rule 3.1, 3.3
    by_day = {day: [] for day in range(1, 8)}
    for item in doc:
        if not isinstance(item, dict):
            return None, "GOOGLE_BAD_ELEMENT"
        src = item.get("day_of_week")
        if not _int(src) or not 0 <= src <= 6:
            return None, "GOOGLE_BAD_DAY"
        if not all(_int(item.get(key)) for key in TIME_KEYS):
            return None, "GOOGLE_BAD_TIME"
        day = 7 if src == 0 else src                                 # rule 1; close_day not read
        by_day[day].append(_open(day, f"{item['open_hour']:02d}:{item['open_minute']:02d}",
                                 f"{item['close_hour']:02d}:{item['close_minute']:02d}"))   # rules 4, 5, 7, 9
    return _dump([obj for day in range(1, 8) for obj in (by_day[day] or [_closed(day)])]), None   # rule 6


def partner_hours(text):
    if text is None:
        return None, None                                            # §1.5
    try:
        doc = json.loads(text)
    except (TypeError, ValueError, RecursionError):
        return None, "PARTNER_INVALID_JSON"
    if not isinstance(doc, dict):
        return None, "PARTNER_NOT_OBJECT"
    days = []
    for day, key in enumerate(PARTNER_DAYS, start=1):
        if key not in doc:
            days.append(_closed(day))                                # no day key
            continue
        entry = doc[key]
        if not isinstance(entry, dict):
            return None, "PARTNER_BAD_DAY"
        if "hours" not in entry or entry["hours"] == []:
            days.append(_closed(day))                                # no / empty hours; status not read
            continue
        hours = entry["hours"]
        if not isinstance(hours, list):
            return None, "PARTNER_BAD_HOURS"
        if len(hours) > 1:
            return None, "PARTNER_MULTI_HOURS"                       # §1.5 stop
        item = hours[0]
        if not isinstance(item, dict) or not isinstance(item.get("open_time"), str) \
                or not isinstance(item.get("close_time"), str):
            return None, "PARTNER_BAD_HOURS"
        days.append(_open(day, item["open_time"], item["close_time"]))
    return _dump(days), None


def operating_hours(source_name, text):
    source = (source_name or "").strip().lower()
    if source == "google":
        return google_hours(text)
    if source == "partner":
        return partner_hours(text)
    return None, "SOURCE_NOT_GOOGLE_PARTNER"
# END_HOURS_PURE


In [ ]:
ENT_COLS = ("poi_id", "poi_name", "source_name", "business_sector", "business_category", "types_json", "phone",
            "email", "website_url", "operating_status", "opening_hours_text", "holiday_exceptions")
DEST_COLS = ("poi_id", "updated_at", "destination_id", "full_address", "street", "ward", "district", "city",
             "country_code", "postal_code", "lat", "lng")
# OUT_COLS của NB_SLV_POI_ENTITY_NON_ENRICH. Cùng tên với cột destination thì mới lấy.
ENTITY_OUT_COLS = ("poi_id", "source_name", "source_id", "language_code", "poi_name", "poi_name_normalized",
                   "types_json", "operating_status", "primary_type", "website_url", "house_number", "street",
                   "ward", "city", "full_address", "lat", "lng", "business_sector", "business_category",
                   "business_category_vi", "category_group", "range_low_min", "range_high_min", "avg_range_min",
                   "candidate_best_time_code", "canonical_candidate", "opening_hours_text", "phone", "email",
                   "holiday_exceptions")
FROM_ENTITY = tuple(name for name in DEST_COLS[2:] if name in ENTITY_OUT_COLS)
RATING_COLS = ("poi_id", "rating_overall", "rating_count")
INT_MIN, INT_MAX = -2147483648, 2147483647
OUT_COLS = ("poi_id", "poi_name", "source_name", "business_sector", "business_category", "subcategory_tags",
            "destination_id", "full_address", "street", "ward", "district", "city", "country_code", "postal_code",
            "lat", "lng", "phone", "email", "website_url", "operating_status", "is_verified", "is_recommended",
            "is_bookable", "operating_hours", "holiday_exceptions", "rating_overall", "rating_count", "review_count")
HOURS_TYPE = T.StructType([T.StructField("value", T.StringType()), T.StructField("error", T.StringType())])
_hours = F.udf(lambda source, text: operating_hours(source, text), HOURS_TYPE)
SRC = F.lower(F.trim(F.col("source_name")))


def _require(df, cols, name, hint=""):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ConfigError(f"gld_srv_poi_info: {name} thiếu {missing}{hint}")


def _destination(des, keys):
    # D-SRV-21: updated_at lớn nhất; trùng mà updated_at NULL hoặc bằng nhau -> dừng, không chọn dòng.
    w = Window.partitionBy("poi_id")
    d = (des.join(keys, "poi_id", "left_semi")
         .withColumn("_n", F.count(F.lit(1)).over(w))
         .withColumn("_nulls", F.count(F.when(F.col("updated_at").isNull(), 1)).over(w))
         .withColumn("_max", F.max("updated_at").over(w))
         .withColumn("_top", F.count(F.when(F.col("updated_at") == F.col("_max"), 1)).over(w)))
    bad = d.where("_n > 1 AND (_nulls > 0 OR _top > 1)").select("poi_id").distinct().count()
    if bad:
        raise QualityGateError(f"gld_srv_poi_info: slv_enr_poi_destination {bad} poi_id trùng, updated_at NULL hoặc bằng nhau")
    ctx_dup = d.where("_n > 1").select("poi_id").distinct().count()
    return d.where("_n = 1 OR updated_at = _max").drop("_n", "_nulls", "_max", "_top", "updated_at"), ctx_dup


def _rating(rating, keys):
    # D-01: một dòng mỗi poi_id. Trùng poi_id -> dừng, không chọn dòng.
    r = rating.select(*RATING_COLS).join(keys, "poi_id", "left_semi")
    dup = r.groupBy("poi_id").count().where("count > 1").count()
    if dup:
        raise QualityGateError(f"gld_srv_poi_info: slv_3p_poi_rating {dup} poi_id có hơn một dòng")
    return r.withColumn("_rated", F.lit(True))


def _reviews(review, keys):
    # D-02: COUNT(*) theo poi_id, review_id NULL vẫn đếm. Không có dòng -> NULL sau left join, không coalesce 0.
    return (review.select("poi_id").join(keys, "poi_id", "left_semi")
            .groupBy("poi_id").agg(F.count(F.lit(1)).alias("review_count")))


def build(ctx):
    ent = ctx.input("entity", live_only=False)
    des = ctx.input("destination", live_only=False)
    ml = ctx.input("multi_lang", live_only=False)
    rt = ctx.input("rating")
    rv = ctx.input("review")
    _require(ent, ENT_COLS + FROM_ENTITY, "slv_poi_entity_non_enrich")
    _require(des, DEST_COLS, "slv_enr_poi_destination", ": chạy srvpoipy của NB_CREATE_DDL")
    _require(ml, ("poi_id",), "slv_enr_poi_info_multi_lang")
    _require(rt, RATING_COLS, "slv_3p_poi_rating")
    _require(rv, ("poi_id",), "slv_3p_poi_review")
    rating_type = dict(rt.dtypes)["rating_overall"]
    if rating_type != "decimal(4,2)":
        raise ConfigError(f"gld_srv_poi_info: slv_3p_poi_rating.rating_overall là {rating_type}, cần decimal(4,2)")
    ent = ent.select(*ENT_COLS, *[F.col(name).alias(f"_e_{name}") for name in FROM_ENTITY])
    des, dup = _destination(des.select(*DEST_COLS), ent.select("poi_id"))
    des = des.withColumn("_has_dest", F.lit(True))
    both = (ent.select("poi_id").join(des.select("poi_id"), "poi_id", "left_semi")
            .join(ml.select("poi_id").distinct(), "poi_id", "left_semi").withColumn("_both", F.lit(True)))
    is_google = F.coalesce(SRC == "google", F.lit(False))
    is_partner = F.coalesce(SRC == "partner", F.lit(False))
    rows = ent.join(both, "poi_id", "left").where(~is_google | F.col("_both").isNotNull()).join(des, "poi_id", "left")
    google = rows.where(is_google).select("poi_id")
    rows = (rows.join(_rating(rt, google), "poi_id", "left").join(_reviews(rv, google), "poi_id", "left")
            .withColumn("_h", _hours(F.col("source_name"), F.col("opening_hours_text"))))
    bad = rows.agg(*[F.sum(F.when(~F.col(c).between(INT_MIN, INT_MAX), 1).otherwise(0)).alias(c)
                     for c in ("rating_count", "review_count")]).first().asDict()
    over = {c: n for c, n in bad.items() if n}
    if over:
        raise QualityGateError("gld_srv_poi_info: ngoài miền INT " + json_dumps(over))
    # D-SRV-22: partner không có dòng destination. Có dòng thì giữ cột destination, kể cả null.
    partner_without_dest = is_partner & F.col("_has_dest").isNull()
    dest_type = dict(des.dtypes)
    from_dest = []
    for name in DEST_COLS[2:]:
        filled = F.col(f"_e_{name}") if name in FROM_ENTITY else F.lit(None)
        from_dest.append(F.when(partner_without_dest, filled.cast(dest_type[name])).otherwise(F.col(name)).alias(name))
    out = rows.select(
        *ENT_COLS[:5], F.col("types_json").alias("subcategory_tags"), *from_dest, "phone", "email",
        "website_url", "operating_status", F.lit(None).cast("boolean").alias("is_verified"),
        is_partner.alias("is_recommended"),
        F.lit(None).cast("boolean").alias("is_bookable"), F.col("_h.value").alias("operating_hours"),
        "holiday_exceptions", F.when(is_google, F.col("rating_overall")).alias("rating_overall"),
        F.when(is_google, F.col("rating_count")).cast("int").alias("rating_count"),
        F.when(is_google, F.col("review_count")).cast("int").alias("review_count"),
        F.when(is_google, F.col("_rated")).alias("_rated"), F.col("_h.error").alias("_err"))
    errors = {r["_err"]: r["n"] for r in out.where("_err IS NOT NULL").groupBy("_err").count()
              .withColumnRenamed("count", "n").collect()}
    if errors:
        raise QualityGateError("gld_srv_poi_info: operating_hours " + json_dumps(errors))
    out = out.drop("_err")
    ctx.metrics.update(destination_dup_poi=dup, partner_rows=out.where(SRC == "partner").count(),
                       google_rows_kept=out.where(SRC == "google").count(),
                       operating_hours_null=out.where("operating_hours IS NULL").count(),
                       rating_matched=out.where("_rated IS NOT NULL").count(),
                       review_matched=out.where("review_count IS NOT NULL").count())
    return out.select(*OUT_COLS)


SPEC = NodeSpec(
    trg_tbl="gld_srv_poi_info",
    inputs={"entity": "slv_poi_entity_non_enrich", "destination": "slv_enr_poi_destination",
            "multi_lang": "slv_enr_poi_info_multi_lang", "rating": "slv_3p_poi_rating",
            "review": "slv_3p_poi_review"},
    pk=("poi_id",),
    build=build,
    required_inputs=("entity",),
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
